In [1]:
import os
import glob
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# 1. Locate and Load the Dataset
# -------------------------------------------------------------
# Find the exact CSV file mounted in /kaggle/input
file_matches = glob.glob('/kaggle/input/**/*.csv', recursive=True)

if not file_matches:
    raise FileNotFoundError("No CSV file found in /kaggle/input. Verify the dataset is attached.")

raw_path = file_matches[0]
print(f"Loading raw data from: {raw_path}")

df_raw = pd.read_csv(raw_path)
print(f"Raw shape: {df_raw.shape[0]:,} rows | {df_raw.shape[1]} columns")

# Standardize column headers
df_raw.columns = df_raw.columns.str.strip().str.lower().str.replace(' ', '_')

# -------------------------------------------------------------
# 2. Filter to Virginia & Active 'for_sale' Listings
# -------------------------------------------------------------
# Standardize state text and slice
df_va = df_raw[df_raw['state'].astype(str).str.strip().str.title() == 'Virginia'].copy()

# Keep active residential sales (filters out pending land/construction tracts)
if 'status' in df_va.columns:
    df_va = df_va[df_va['status'].astype(str).str.lower().str.contains('for_sale', na=False)]

print(f"Virginia 'for_sale' listings: {len(df_va):,} rows")

# -------------------------------------------------------------
# 3. Clean Missing Core Values & Standardize Formats
# -------------------------------------------------------------
# Drop listings missing baseline price, size, or location values
df_va = df_va.dropna(subset=['price', 'house_size', 'zip_code', 'city'])

# Format zip codes: ensure 5-digit string format (e.g., handles 23602.0 -> "23602")
df_va['zip_code'] = (
    df_va['zip_code']
    .astype(float)
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

# Standardize city names
df_va['city'] = df_va['city'].astype(str).str.strip().str.title()

# -------------------------------------------------------------
# 4. Deduplicate Across Scrapes & Filter Realistic Bounds
# -------------------------------------------------------------
# Deduplicate properties repeated across scrape runs
dedup_cols = ['street', 'city', 'zip_code', 'price', 'house_size', 'bed', 'bath']
existing_dedup = [c for c in dedup_cols if c in df_va.columns]
df_va = df_va.drop_duplicates(subset=existing_dedup, keep='last')

# Filter residential bounds (removes commercial tracts, $0 placeholders, and entry typos)
cleaned_df = df_va[
    (df_va['price'] >= 50_000) & (df_va['price'] <= 5_000_000) &
    (df_va['house_size'] >= 400) & (df_va['house_size'] <= 15_000) &
    (df_va['bed'] >= 1) & (df_va['bed'] <= 10) &
    (df_va['bath'] >= 1) & (df_va['bath'] <= 10)
].copy()

# -------------------------------------------------------------
# 5. Feature Engineering: Price per Sq Ft & Ratios
# -------------------------------------------------------------
cleaned_df['price_per_sqft'] = (cleaned_df['price'] / cleaned_df['house_size']).round(2)
cleaned_df['bed_bath_ratio'] = (cleaned_df['bed'] / cleaned_df['bath']).round(2)

# Impute missing lot acreage using the median lot size for that zip code
if 'acre_lot' in cleaned_df.columns:
    zip_median_lot = cleaned_df.groupby('zip_code')['acre_lot'].transform('median')
    cleaned_df['acre_lot'] = cleaned_df['acre_lot'].fillna(zip_median_lot).fillna(0.25)

# Drop redundant or extraneous columns
drop_cols = ['brokered_by', 'state', 'prev_sold_date']
cleaned_df = cleaned_df.drop(columns=[c for c in drop_cols if c in cleaned_df.columns])

print("\n--- Cleaning Complete ---")
print(f"Final Cleaned Virginia Dataset: {cleaned_df.shape[0]:,} rows | {cleaned_df.shape[1]} columns")
print(cleaned_df[['city', 'zip_code', 'price', 'house_size', 'price_per_sqft']].head())

# -------------------------------------------------------------
# 6. Save Clean Slice to Kaggle Working Directory
# -------------------------------------------------------------
output_path = "/kaggle/working/cleaned_va_real_estate.csv"
cleaned_df.to_csv(output_path, index=False)
print(f"\nSaved successfully to: {output_path}")

Loading raw data from: /kaggle/input/datasets/ahmedshahriarsakib/usa-real-estate-dataset/realtor-data.zip.csv
Raw shape: 2,226,382 rows | 12 columns
Virginia 'for_sale' listings: 35,572 rows

--- Cleaning Complete ---
Final Cleaned Virginia Dataset: 23,437 rows | 11 columns
                  city zip_code     price  house_size  price_per_sqft
183565    Lovettsville    20180  850000.0      4620.0          183.98
185183  Cross Junction    22625  399000.0      2000.0          199.50
185186      Winchester    22601  345000.0      1550.0          222.58
185187   Stephens City    22655  299900.0      1254.0          239.15
185194      Winchester    22602  385000.0      2033.0          189.38

Saved successfully to: /kaggle/working/cleaned_va_real_estate.csv
